# الدرس ١: قراءة ملف PDF 📄

## الهدف
في آخر الدرس دا حيكون عندك **function** بتقرأ أي تقرير مالي (PDF) وبترجّع نص كل صفحة مع رقمها.

## ليه دي أول خطوة؟
نظام RAG بيجاوب على الأسئلة من **نص** المستندات. فقبل أي بحث أو ذكاء اصطناعي، لازم نطلّع النص من الـ PDF.

## الأداة: PyMuPDF
مكتبة اسمها `pymupdf`، لكن بنعمل ليها import باسم `fitz` (اسم قديم، ما تستغرب).

| الكود | بيعمل شنو |
|---|---|
| `doc = fitz.open(path)` | يفتح الملف |
| `len(doc)` | عدد الصفحات |
| `doc[0]` | أول صفحة (**الترقيم بيبدأ من 0**) |
| `page.get_text()` | نص الصفحة |

> **طريقة الشغل:** شغّل الخلايا بالترتيب. في كل مكان مكتوب فيه `# TODO` إنت بتكتب الكود. لو علقت، اسألني. ما في إجابات جاهزة هنا 😉

In [ ]:
# Step 0: imports (already done for you)
import fitz
from pathlib import Path

PDF_PATH = Path("../data/raw/3M_2018_10K.pdf")

print("File exists:", PDF_PATH.exists())

---
## تمرين ١: افتح الملف واعرف عدد صفحاته

**المطلوب:** افتح الملف في متغير اسمه `doc`، واطبع عدد الصفحات.

💡 **تلميح:** شوف الجدول فوق.

In [ ]:
# TODO: open the PDF
doc = ...

# TODO: print the number of pages

---
## تمرين ٢: اطبع نص أول صفحة

**المطلوب:** اطبع أول 500 حرف من نص **الصفحة الأولى**.

💡 **تلميح:** أول صفحة رقمها `0` في الكود. ولأخذ أول 500 حرف من نص: `text[:500]`.

🤔 **لاحظ:** النص شكله كيف؟ فيه سطور فاضية؟ رموز غريبة؟

In [ ]:
# TODO: get the first page
first_page = ...

# TODO: get its text and print the first 500 characters

---
## تمرين ٣: ابحث عن معلومة في التقرير

السؤال الحقيقي من FinanceBench:
> *What is the FY2018 capital expenditure amount for 3M?*

في التقارير المالية، الـ capital expenditure بتظهر كسطر اسمه **"Purchases of property, plant and equipment"**.

**المطلوب:** لف على كل الصفحات، واطبع **رقم كل صفحة** فيها الجملة دي.

💡 **تلميحات:**
- `for i, page in enumerate(doc):` بتديك رقم الصفحة `i` والصفحة نفسها.
- استخدم `.lower()` على النص وعلى الجملة، عشان الحروف الكبيرة والصغيرة ما تفرق.
- اطبع `i + 1` بدل `i`. **ليه؟** فكّر قبل ما تقرأ تمرين ٤.

In [ ]:
SEARCH_PHRASE = "purchases of property, plant and equipment"

found_pages = []

# TODO: loop over all pages; if SEARCH_PHRASE is in the page text,
#       add the page number (starting from 1) to found_pages


print("Found on pages:", found_pages)

---
## تمرين ٤: اكتب الـ function النهائية

دي أهم خطوة: نحوّل اللي عملناه لـ **function** بنستخدمها في كل الدروس الجاية.

**المطلوب:** الـ function ترجّع **list**، كل عنصر فيها **dict** بالشكل دا:

```python
{
    "page": 1,                  # رقم الصفحة، بيبدأ من 1 (زي ما الإنسان بيعد)
    "text": "UNITED STATES...", # نص الصفحة
    "source": "3M_2018_10K.pdf" # اسم الملف
}
```

💡 **تلميح:** `pdf_path.name` بتديك اسم الملف.

**ليه بنحفظ رقم الصفحة واسم الملف؟** عشان لما النظام يجاوب، يقدر يقول **"لقيت الإجابة في صفحة 60 من تقرير 3M"**. دا اسمه **citation**، ومهم جداً في الأنظمة المالية.

In [ ]:
def extract_pages(pdf_path: Path) -> list[dict]:
    """
    Read a PDF and return one dict per page:
    {"page": int (starting at 1), "text": str, "source": str}
    """

    pages = []

    # TODO: open the PDF, loop over its pages, and append one dict per page

    return pages


pages = extract_pages(PDF_PATH)

print("Pages:", len(pages))
print(pages[59]["page"], pages[59]["source"])
print(pages[59]["text"][:300])

---
## ✅ اختبر شغلك

شغّل الخلية دي. لو كل حاجة صاح حتطلع ليك ✅ في كل سطر.

In [ ]:
def check(name, condition):
    print(("✅ " if condition else "❌ ") + name)

check("doc has 160 pages", len(doc) == 160)
check("found_pages contains page 60 (the cash flow statement)", 60 in found_pages)
check("extract_pages returns 160 pages", len(pages) == 160)
check("page numbers start at 1", pages[0]["page"] == 1)
check("each page has text, page and source",
      all(set(p) == {"page", "text", "source"} for p in pages))
check("source is the file name", pages[0]["source"] == "3M_2018_10K.pdf")

---
## 🧠 أسئلة للتفكير

جاوب عليها بكلامك قبل ما نمشي للدرس الجاي. اكتبها في الخلية تحت أو قولها لي:

1. ليه الكود بيبدأ الترقيم من `0`، ونحنا بنحفظ الصفحة من `1`؟
2. في تمرين ٣ لقيت الجملة في **أكتر من صفحة**. ليه دا ممكن يكون مشكلة لو عايزين نلقى **الإجابة الصحيحة**؟
3. لو الـ PDF عبارة عن **صورة ممسوحة** (scan)، `get_text()` حترجّع شنو؟

**الدرس الجاي:** تقسيم النص لقطع صغيرة (chunking)، وليه ما بنستخدم الصفحة كاملة.

### إجاباتي:

1. 
2. 
3. 